# IndoBERTweet + weighted loss — dataset **v1audited** (Track C)

Coba angkat akurasi di atas IndoBERT v1audited (0,8633). Model
`indolem/indobertweet-base-uncased` (dilatih Twitter Indonesia, cocok bahasa medsos) +
weighted cross-entropy. **Label dari `balanced_3000_v1audited.csv`** (clone repo via PAT);
teks `bert` dari Mongo `processed_bert` (label-independent). Split kanonik identik SVM/IndoBERT
(urut comment_id, seed=42, 70/20/10). Mongo TIDAK diubah.

**Sebelum Run all:** Runtime → **T4 GPU**. Secret `GITHUB_PAT` + `MONGO_URI` (atau ketik manual).
Output `indobertweet_v1audited_metrics.json` ter-download.

Pembanding: set `MODEL_NAME='indobenchmark/indobert-base-p1'` utk ukur efek weighted-loss saja.

In [ ]:
!nvidia-smi -L || echo 'Aktifkan T4: Runtime > Change runtime type > GPU'
%pip install -q "transformers>=4.40" torch "pymongo[srv]" dnspython certifi scikit-learn matplotlib pandas numpy

In [ ]:
# --- Secrets + clone repo (ambil CSV v1audited) ---
import os, subprocess, shutil, pathlib
def _secret(name):
    v = os.environ.get(name, '')
    if not v:
        try:
            from google.colab import userdata; v = userdata.get(name) or ''
        except Exception: v = ''
    if not v:
        from getpass import getpass; v = getpass(f'{name}: ')
    return v

GITHUB_PAT = _secret('GITHUB_PAT')
os.environ['MONGO_URI'] = _secret('MONGO_URI')
REPO = '/content/jokowi_sentiment_project'
if pathlib.Path(REPO).exists(): shutil.rmtree(REPO)
r = subprocess.run(['git','clone','--depth','1',f'https://{GITHUB_PAT}@github.com/Arachnoida/jokowi_sentiment_project.git',REPO],
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('clone OK' if r.returncode==0 else 'CLONE GAGAL:')
print('\n'.join(l for l in r.stdout.splitlines() if 'github.com' not in l))
assert r.returncode==0
CSV = f'{REPO}/outputs/labeling/balanced_3000_v1audited.csv'
assert pathlib.Path(CSV).exists(), 'CSV v1audited tak ada di repo'

In [ ]:
# --- bert dari Mongo + label dari CSV v1audited + split kanonik ---
import pandas as pd, certifi
from pymongo import MongoClient
from sklearn.model_selection import train_test_split

MODEL_NAME = 'indolem/indobertweet-base-uncased'   # pembanding: 'indobenchmark/indobert-base-p1'
TAG = 'v1audited'
LABELS = ['Negatif','Netral','Positif']; LABEL2ID = {l:i for i,l in enumerate(LABELS)}
DB = os.environ.get('MONGO_DB_NAME','youtube_sentiment')

client = MongoClient(os.environ['MONGO_URI'], tlsCAFile=certifi.where(), serverSelectionTimeoutMS=30000)
client.admin.command('ping'); print('Mongo OK.')
sub = pd.read_csv(CSV); sub['comment_id'] = sub['comment_id'].astype(str)
bert = pd.DataFrame(list(client[DB]['processed_bert'].find({}, {'_id':0,'comment_id':1,'bert':1})))
bert['comment_id'] = bert['comment_id'].astype(str)
df = sub[['comment_id','label']].merge(bert, on='comment_id', how='left')   # label dari CSV
df['bert'] = df['bert'].fillna('')
df['label_id'] = df['label'].map(LABEL2ID)
assert df['label_id'].notna().all() and (df['bert'].str.len()>0).all(), 'ada label/bert kosong'
df = df.sort_values('comment_id').reset_index(drop=True)
tmp, df_test = train_test_split(df, test_size=0.10, stratify=df['label_id'], random_state=42)
df_train, df_val = train_test_split(tmp, test_size=0.20/0.90, stratify=tmp['label_id'], random_state=42)
print(f'{MODEL_NAME} | {TAG} | n={len(df)} train={len(df_train)} val={len(df_val)} test={len(df_test)}')
print(df['label'].value_counts().to_string())

In [ ]:
# --- Tokenizer + Dataset ---
from transformers import AutoTokenizer
import torch
MAX_LEN, SEED = 128, 42
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True); self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        it = {k: torch.tensor(v[i]) for k, v in self.enc.items()}; it['labels'] = torch.tensor(self.labels[i]); return it
ds_train = DS(df_train['bert'].astype(str), df_train['label_id'])
ds_val   = DS(df_val['bert'].astype(str),   df_val['label_id'])
ds_test  = DS(df_test['bert'].astype(str),  df_test['label_id'])

In [ ]:
# --- Model + weighted loss + Trainer ---
import numpy as np
from collections import Counter
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score
set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label={i:l for i,l in enumerate(LABELS)}, label2id=LABEL2ID)
cnt = Counter(df_train['label_id']); n = len(df_train)
class_w = torch.tensor([n/(3*cnt[i]) for i in range(3)], dtype=torch.float)
print('class weights:', [round(x,3) for x in class_w.tolist()])
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=class_w.to(out.logits.device))
        return (loss, out) if return_outputs else loss
def compute_metrics(p):
    pr = np.argmax(p.predictions, axis=1)
    return {'macro_f1': f1_score(p.label_ids, pr, average='macro'), 'accuracy': accuracy_score(p.label_ids, pr)}
_kw = dict(output_dir='out', num_train_epochs=6, per_device_train_batch_size=16, per_device_eval_batch_size=32,
           learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1, save_total_limit=2,
           load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
           seed=SEED, logging_steps=50, report_to='none')
try:
    args = TrainingArguments(eval_strategy='epoch', save_strategy='epoch', **_kw)
except TypeError:
    args = TrainingArguments(evaluation_strategy='epoch', save_strategy='epoch', **_kw)
trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                          compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
trainer.train(); print('Val terbaik:', trainer.evaluate())

In [ ]:
# --- Evaluasi test + simpan + download ---
import json
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(ds_test)
y_pred = np.argmax(pred.predictions, axis=1).tolist(); y_true = df_test['label_id'].tolist()
def evaluate(yt, yp, labels=LABELS):
    ids = list(range(len(labels)))
    rep = classification_report(yt, yp, labels=ids, target_names=labels, output_dict=True, zero_division=0)
    return {'accuracy': round(accuracy_score(yt,yp),4), 'macro_f1': round(f1_score(yt,yp,average='macro',zero_division=0),4),
            'weighted_f1': round(f1_score(yt,yp,average='weighted',zero_division=0),4),
            'per_class': {l:{k:round(rep[l][k],4) for k in ['precision','recall','f1-score']} | {'support':int(rep[l]['support'])} for l in labels},
            'confusion_matrix': confusion_matrix(yt,yp,labels=ids).tolist(), 'labels': labels}
m = evaluate(y_true, y_pred)
print('='*56); print(f'  {MODEL_NAME} [{TAG}] TEST'); print('='*56)
print(f"  Accuracy : {m['accuracy']:.4f}  | macro-F1: {m['macro_f1']:.4f}  (IndoBERT v1audited=0.8633)")
for l in LABELS:
    pc = m['per_class'][l]; print(f"    {l:<8} P={pc['precision']:.3f} R={pc['recall']:.3f} F1={pc['f1-score']:.3f}")
json.dump({'model':MODEL_NAME,'dataset':TAG,'test':m}, open('indobertweet_v1audited_metrics.json','w'), ensure_ascii=False, indent=2)
logits = np.asarray(pred.predictions, dtype=float); e = np.exp(logits - logits.max(axis=1, keepdims=True))
pdf = pd.DataFrame({'comment_id': df_test['comment_id'].to_numpy(), 'text': df_test['bert'].to_numpy(),
                    'label_asli': df_test['label'].to_numpy(), 'prediksi': [LABELS[i] for i in y_pred]})
pdf['benar'] = pdf['label_asli'] == pdf['prediksi']; pdf['keyakinan'] = np.round((e/e.sum(axis=1,keepdims=True)).max(axis=1),4)
pdf.to_csv('indobertweet_v1audited_predictions.csv', index=False)
import matplotlib.pyplot as plt
cm = np.array(m['confusion_matrix']); fig, ax = plt.subplots(figsize=(5,4.3)); ax.imshow(cm, cmap='Purples')
ax.set_xticks(range(3), LABELS); ax.set_yticks(range(3), LABELS); ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
ax.set_title(f"IndoBERTweet [{TAG}] (acc={m['accuracy']:.3f})")
th = cm.max()/2
for i in range(3):
    for j in range(3): ax.text(j,i,cm[i,j],ha='center',va='center',color='white' if cm[i,j]>th else 'black')
fig.tight_layout(); fig.savefig('indobertweet_v1audited_test_confusion.png', dpi=120); plt.show()
try:
    from google.colab import files
    files.download('indobertweet_v1audited_metrics.json')
    files.download('indobertweet_v1audited_test_confusion.png')
    files.download('indobertweet_v1audited_predictions.csv')
except Exception as ex:
    print('Download manual dari panel Files:', ex)